In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
import torch
from dataset import load_ohio_data
from model   import GlucoseTransformer, TransformerConfig
from trainer import Trainer, TrainingConfig

TRAIN_DIR = "OhioT1DM/2020/train/"
TEST_DIR  = "OhioT1DM/2020/test/"
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Load data ─────────────────────────────────────────────────────────────────
split = load_ohio_data(
    train_dir   = TRAIN_DIR,
    test_dir    = TEST_DIR,
    batch_size  = 64,
    num_workers = 0,
    verbose     = False,
)

# ── Config ────────────────────────────────────────────────────────────────────
model_config = TransformerConfig()
train_config = TrainingConfig()

# ── Build model ───────────────────────────────────────────────────────────────
model = GlucoseTransformer(model_config).to(DEVICE)
print(f"Device          : {DEVICE}")
print(f"Model parameters: {model.count_parameters():,}")
print(f"Train sequences : {len(split.train_loader.dataset):,}")
print(f"Val sequences   : {len(split.val_loader.dataset):,}")

# ── Train ─────────────────────────────────────────────────────────────────────
trainer = Trainer(model, train_config, model_config)
history = trainer.fit(split.train_loader, split.val_loader)

Patient 540: 11649 valid sequences from 13110 grid timesteps (glucose mean=137.1, std=55.0)
Patient 540: 2842 valid sequences from 3067 grid timesteps (glucose mean=137.1, std=55.0)
Patient 544: 10190 valid sequences from 12672 grid timesteps (glucose mean=165.2, std=59.9)
Patient 544: 2610 valid sequences from 3137 grid timesteps (glucose mean=165.2, std=59.9)
Patient 552: 8707 valid sequences from 11098 grid timesteps (glucose mean=146.7, std=54.6)
Patient 552: 2286 valid sequences from 3951 grid timesteps (glucose mean=146.7, std=54.6)
Patient 567: 10038 valid sequences from 13537 grid timesteps (glucose mean=153.8, std=60.9)
  Patient 567: 'meal' stream absent — no carb features for this file
Patient 567: 2249 valid sequences from 2872 grid timesteps (glucose mean=153.8, std=60.9)
Patient 584: 11803 valid sequences from 13249 grid timesteps (glucose mean=192.5, std=65.4)
Patient 584: 2618 valid sequences from 2996 grid timesteps (glucose mean=192.5, std=65.4)
Patient 596: 10430 val

/Users/owenlee/miniconda3/envs/ai2026_good/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

In [3]:
from calibrator import ConformalCalibrator
from model import GlucoseTransformer
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load best model checkpoint
checkpoint = torch.load(
    "checkpoints/best_model.pt",
    map_location = DEVICE,
    weights_only = False,
)
model_config = checkpoint['model_config']
model        = GlucoseTransformer(model_config).to(DEVICE)
model.load_state_dict(checkpoint['model_state'])
model.eval()

print(f"Loaded best model from epoch "
      f"{checkpoint['epoch'] + 1} "
      f"(val loss {checkpoint['val_loss']:.6f})")

# Run conformal calibration on validation set
calibrator = ConformalCalibrator(model, model_config, DEVICE)
results    = calibrator.calibrate(
    cal_loader    = split.val_loader,
    patient_stats = split.patient_stats,
)

# Save calibration results
calibrator.save("checkpoints/calibration.pkl")

Loaded best model from epoch 8 (val loss 0.081435)
Running conformal calibration...
  Quantiles : [0.05, 0.25, 0.5, 0.75, 0.95]
  Calibration samples: 18,848
  Calibrating intervals: ['90%', '50%']


  90% interval (q5/q95):
    Calibration samples : 18,848
    Raw coverage        : 0.9192  (target 0.90, gap +0.0192)
    Adjusted coverage   : 0.9000  (gap +0.0000)
    q_hat (normalized)  : -0.035023
    q_hat (mg/dL, mean) : -2.01
    Range-specific coverage after adjustment:
      hypo      : 0.6633  (n=496)
      normal    : 0.9109  (n=12,053)
      hyper     : 0.8979  (n=6,299)

  50% interval (q25/q75):
    Calibration samples : 18,848
    Raw coverage        : 0.5051  (target 0.50, gap +0.0051)
    Adjusted coverage   : 0.5001  (gap +0.0001)
    q_hat (normalized)  : -0.001946
    q_hat (mg/dL, mean) : -0.11
    Range-specific coverage after adjustment:
      hypo      : 0.2581  (n=496)
      normal    : 0.5103  (n=12,053)
      hyper     : 0.4994  (n=6,299)
Calibration results sa

In [6]:
import importlib
import calibrator
importlib.reload(calibrator)
from calibrator import ConformalCalibrator

In [8]:
range_results = calibrator.calibrate_by_range(
    cal_loader = split.val_loader,
)
calibrator.save("checkpoints/calibration.pkl")

AttributeError: module 'calibrator' has no attribute 'calibrate_by_range'